# Multimodal — encodeur-décodeur et rééchantillonnage

Essais d'architecture encodeur-décodeur et de plusieurs méthodes de rééchantillonnage (SMOTE, ADASYN, SMOTE-Tomek).

**Statut :** Exploratoire

*Mémoire de maîtrise — Mireille Kafando, UQTR · mai 2025*

> Notebook de recherche conçu pour Google Colab. Les chemins `/content/drive/MyDrive/bug/...` sont à adapter (voir `data/README.md`). Les sorties ont été retirées.


# Mounting drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter

# Fix pour erreur avec TF
os.environ['TRANSFORMERS_NO_TF'] = '1'

# Configuration améliorée
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 5
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5  # Augmenté pour plus de stabilité
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Nouveaux paramètres pour l'équilibrage
    CLASS_WEIGHTS = None  # Sera calculé dynamiquement
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé amélioré
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

        # Vérifier la cohérence des tailles
        assert len(self.metrics) == len(self.labels), "Mismatch entre metrics et labels"
        assert len(self.code_tokens['input_ids']) == len(self.labels), "Mismatch entre code_tokens et labels"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# Modèle hybride amélioré avec normalisation et régularisation
class ImprovedTransBugModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()
        # Projection des métriques avec normalisation
        self.metric_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM) if input_metric_dim is None else nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder pour les métriques avec normalisation
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBEDDING_DIM,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            activation='gelu',  # GELU au lieu de ReLU
            batch_first=True,
            norm_first=True  # Pre-normalization
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        # Décodeur RoBERTa pré-entraîné avec gradient scaling
        roberta_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(roberta_config)

        # Réduire le learning rate pour les couches pré-entraînées
        for param in self.code_decoder.parameters():
            param.requires_grad = True

        # Projection pour aligner les dimensions
        self.code_projection = nn.Linear(roberta_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention croisée améliorée
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion avec skip connection
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final avec couches intermédiaires
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques avec normalisation
        metrics_proj = self.metric_projection(metrics)
        metrics_proj = metrics_proj.unsqueeze(1)  # [B, 1, 384]
        metrics_encoded = self.encoder(metrics_proj)  # [B, 1, 384]

        # Traitement du code source
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_embed = self.code_projection(code_embed)  # [B, seq_len, 384]

        # Attention croisée
        attn_output, attn_weights = self.cross_attention(
            metrics_encoded,  # Query [B, 1, 384]
            code_embed,      # Key [B, seq_len, 384]
            code_embed       # Value [B, seq_len, 384]
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# Stratégies d'équilibrage améliorées
def apply_advanced_sampling(X, y, code_lines, strategy='smote_tomek'):
    """
    Applique différentes stratégies d'équilibrage des classes
    """
    print(f"Distribution avant équilibrage: {Counter(y)}")

    if strategy == 'smote':
        sampler = SMOTE(random_state=42, k_neighbors=3)
    elif strategy == 'borderline_smote':
        sampler = BorderlineSMOTE(random_state=42, k_neighbors=3)
    elif strategy == 'adasyn':
        sampler = ADASYN(random_state=42, n_neighbors=3)
    elif strategy == 'smote_tomek':
        sampler = SMOTETomek(random_state=42, smote=SMOTE(k_neighbors=3))
    elif strategy == 'smote_enn':
        sampler = SMOTEENN(random_state=42, smote=SMOTE(k_neighbors=3))
    else:
        return X, y, code_lines

    # Appliquer l'équilibrage sur les métriques
    X_resampled, y_resampled = sampler.fit_resample(X, y)

    print(f"Distribution après équilibrage: {Counter(y_resampled)}")

    # Adapter la liste de code
    n_original = len(code_lines)
    n_resampled = len(X_resampled)

    if n_resampled > n_original:
        # Créer un mapping pour les indices répliqués
        indices_map = []
        for i in range(n_resampled):
            if i < n_original:
                indices_map.append(i)
            else:
                # Pour les nouveaux échantillons, choisir aléatoirement
                indices_map.append(np.random.randint(0, n_original))

        # Ajuster la liste de code
        code_lines_resampled = [code_lines[idx] for idx in indices_map]
    else:
        code_lines_resampled = code_lines[:n_resampled]

    return X_resampled, y_resampled, code_lines_resampled

# Calcul des poids de classe
def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    return weight_tensor

# Prétraitement amélioré des métriques
def preprocess_metrics(df):
    """Préprocessing robuste des métriques"""
    # Supprimer les colonnes non-numériques
    if 'name' in df.columns:
        df = df.drop(columns=['name'])

    # Binariser la variable cible
    df['bug'] = (df['bug'] > 0).astype(int)

    # Séparer features et labels
    features = df.drop(columns=['bug'])
    labels = df['bug'].values

    # Vérifier et traiter les valeurs manquantes
    if features.isnull().any().any():
        print("Valeurs manquantes détectées, remplacement par la médiane")
        features = features.fillna(features.median())

    # Supprimer les features avec variance nulle
    var_threshold = 0.01
    low_var_cols = features.var() < var_threshold
    if low_var_cols.any():
        print(f"Suppression de {low_var_cols.sum()} colonnes avec faible variance")
        features = features.loc[:, ~low_var_cols]

    # Normalisation robuste
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    # Vérifier la distribution des classes
    print(f"Distribution des classes: {Counter(labels)}")

    return features_scaled, labels, scaler

# Préparation du dataset avec équilibrage avancé
def prepare_single_dataset(metrics_path, code_path, tokenizer, apply_balancing=True, balancing_strategy='smote_tomek'):
    """Prépare un dataset avec équilibrage avancé"""
    df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
    features, labels, scaler = preprocess_metrics(df_metrics)

    # Assurer la cohérence des tailles
    min_len = min(len(features), len(code_lines), len(labels))
    features = features[:min_len]
    code_lines = code_lines[:min_len]
    labels = labels[:min_len]

    if apply_balancing and len(set(labels)) > 1:  # Vérifier qu'il y a au moins 2 classes
        try:
            features, labels, code_lines = apply_advanced_sampling(
                features, labels, code_lines, strategy=balancing_strategy
            )
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")

    # Tokenisation du code
    try:
        code_tokens = tokenizer(
            code_lines,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        # Fallback: utiliser des tokens vides
        code_tokens = {
            'input_ids': torch.zeros((len(features), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(features), 512), dtype=torch.long)
        }

    return features, code_tokens, labels, scaler

# Chargement des métriques et du code (fonction manquante dans le code original)
def load_code_and_metrics(metrics_path, code_path):
    """Charge les métriques et le code source"""
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []

    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                            code = f.read()
                            # Nettoyer le code
                            code = code.strip()
                            if len(code) > 0:
                                code_lines.append(code)
                    except Exception as e:
                        print(f"Erreur lecture fichier {file}: {e}")
                        continue

    # S'assurer qu'on a au moins autant de fichiers code que de métriques
    if len(code_lines) < len(df_metrics):
        print(f"Attention: {len(code_lines)} fichiers code vs {len(df_metrics)} métriques")
        # Dupliquer des exemples de code si nécessaire
        while len(code_lines) < len(df_metrics):
            if len(code_lines) > 0:
                code_lines.append(code_lines[len(code_lines) % len(code_lines)])
            else:
                code_lines.append("// Empty code file")

    return df_metrics, code_lines

# Évaluation améliorée avec gestion des cas limites
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            # Déplacer les tokens vers le device
            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques de manière robuste
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    # Gestion des cas où une classe est absente
    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)  # Recall
        spec = tn / (tn + fp + 1e-8)  # Specificity

        # G-Mean seulement si sens et spec > 0
        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# Fonction d'entraînement principale améliorée
def train(use_full_pretrained=False, balancing_strategy='smote_tomek'):
    """Entraînement avec équilibrage avancé et monitoring"""
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Définir les chemins
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/ant/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/ant/"

    # Versions pour l'entraînement
    train_versions = ["1.3", "1.4", "1.5", "1.6"]
    metrics_train_paths = [f"{base_metrics_path}ant-{v}.csv" for v in train_versions]
    code_train_paths = [f"{base_code_path}ant-{v}" for v in train_versions]

    # Version pour la validation
    val_version = "1.7"
    metrics_val_path = f"{base_metrics_path}ant-{val_version}.csv"
    code_val_path = f"{base_code_path}ant-{val_version}"

    # Préparer les datasets
    all_train_features, all_train_labels, all_train_code = [], [], []

    for metrics_path, code_path in zip(metrics_train_paths, code_train_paths):
        features, code_tokens, labels, _ = prepare_single_dataset(
            metrics_path, code_path, tokenizer,
            apply_balancing=True, balancing_strategy=balancing_strategy
        )
        all_train_features.append(features)
        all_train_labels.extend(labels)

        # Convertir les tokens en listes pour concaténation
        for i in range(len(features)):
            all_train_code.append({
                'input_ids': code_tokens['input_ids'][i],
                'attention_mask': code_tokens['attention_mask'][i]
            })

    # Concaténer toutes les features d'entraînement
    X_train = np.vstack(all_train_features)
    y_train = np.array(all_train_labels)

    # Reconvertir les tokens en format PyTorch
    train_code_tokens = {
        'input_ids': torch.stack([item['input_ids'] for item in all_train_code]),
        'attention_mask': torch.stack([item['attention_mask'] for item in all_train_code])
    }

    # Préparer le dataset de validation (sans équilibrage)
    X_val, code_val_tokens, y_val, _ = prepare_single_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    # Créer les datasets
    train_ds = BugDataset(X_train, train_code_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)

    # Créer les DataLoaders
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Calculer les poids de classe
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    print(f"Poids des classes: {class_weights}")

    # Initialiser le modèle
    model = ImprovedTransBugModel().to(Config.DEVICE)
    print(f"Modèle initialisé avec stratégie d'équilibrage: {balancing_strategy}")

    # Optimiseur avec différents learning rates
    optimizer = torch.optim.AdamW([
        {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},  # LR plus faible pour pré-entraîné
        {'params': [p for n, p in model.named_parameters() if 'code_decoder' not in n], 'lr': Config.LEARNING_RATE}
    ], weight_decay=0.01)

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=True
    )

    # Loss avec pondération et label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0

    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        print(f"Epoch {epoch+1}:")
        print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
        print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
        print(f"  Val: Sens={sens:.4f}, Spec={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            model_name = f"best_model_{balancing_strategy}.pt"
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}{model_name}")
            print(f"  Nouveau meilleur F1: {f1:.4f}")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 7:
            print("Early stopping déclenché")
            break

    print(f"\nMeilleur F1 score: {best_f1:.4f}")
    print("Classification Report:")
    if len(set(val_labels)) > 1:
        print(classification_report(val_labels, val_preds))
    else:
        print("Une seule classe présente dans les prédictions")

    return {
        'strategy': balancing_strategy,
        'best_f1': best_f1,
        'final_gmean': gmean_scores[-1] if gmean_scores else 0,
        'val_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'train_accuracy': train_accuracies[-1] if train_accuracies else 0,
    }

if __name__ == "__main__":
    # Tester différentes stratégies d'équilibrage
    strategies = ['smote', 'borderline_smote', 'smote_tomek', 'adasyn']
    results = {}

    for strategy in strategies:
        print(f"\n{'='*50}")
        print(f"Test de la stratégie: {strategy}")
        print(f"{'='*50}")

        try:
            result = train(use_full_pretrained=False, balancing_strategy=strategy)
            results[strategy] = result
            print(f"Résultat {strategy}: F1={result['best_f1']:.4f}, G-Mean={result['final_gmean']:.4f}")
        except Exception as e:
            print(f"Erreur avec la stratégie {strategy}: {e}")
            results[strategy] = {'best_f1': 0, 'final_gmean': 0, 'error': str(e)}

    # Résumé des résultats
    print(f"\n{'='*50}")
    print("RÉSUMÉ DES PERFORMANCES")
    print(f"{'='*50}")

    for strategy, result in results.items():
        if 'error' not in result:
            print(f"{strategy:15}: F1={result['best_f1']:.4f}, G-Mean={result['final_gmean']:.4f}")
        else:
            print(f"{strategy:15}: ERREUR - {result['error']}")

    # Trouver la meilleure stratégie
    best_strategy = max(results.keys(), key=lambda k: results[k]['best_f1'] if 'error' not in results[k] else 0)
    print(f"\nMeilleure stratégie: {best_strategy} (F1={results[best_strategy]['best_f1']:.4f})")

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter

# Fix pour erreur avec TF
os.environ['TRANSFORMERS_NO_TF'] = '1'

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

        # Vérifier la cohérence des tailles
        assert len(self.metrics) == len(self.labels), "Mismatch entre metrics et labels"
        assert len(self.code_tokens['input_ids']) == len(self.labels), "Mismatch entre code_tokens et labels"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# Modèle hybride (encoder from scratch + decoder pré-entraîné)
class HybridTransBugModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()
        # Projection des métriques vers la dimension d'embedding
        self.metric_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM) if input_metric_dim is None else nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder transformer from scratch pour les métriques
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBEDDING_DIM,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.metrics_encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        # Decoder pré-entraîné RoBERTa pour le code
        roberta_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(roberta_config)

        # Projection pour aligner les dimensions RoBERTa -> Config.EMBEDDING_DIM
        self.code_projection = nn.Linear(roberta_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention pour combiner les représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques avec encoder from scratch
        metrics_proj = self.metric_projection(metrics)
        metrics_proj = metrics_proj.unsqueeze(1)  # [B, 1, 384]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, 384]

        # Traitement du code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_embed = self.code_projection(code_embed)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_embed,      # Key
            code_embed       # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# Modèle entièrement pré-entraîné (encoder et decoder pré-entraînés)
class FullPretrainedModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder pré-entraîné pour les métriques (adaptation de RoBERTa)
        self.metrics_encoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.metrics_encoder_config.hidden_size = Config.EMBEDDING_DIM
        self.metrics_encoder = RobertaModel(self.metrics_encoder_config)

        # Projection pour les métriques numériques vers le format attendu par transformer
        self.metrics_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Decoder pré-entraîné RoBERTa pour le code
        self.code_decoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(self.code_decoder_config)

        # Projection pour aligner les dimensions
        self.code_projection = nn.Linear(self.code_decoder_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention croisée pour combiner les deux représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Projeter les métriques et les adapter au format attendu
        metrics_embedded = self.metrics_projection(metrics).unsqueeze(1)  # [B, 1, 384]

        # Créer un masque factice pour les métriques (toutes valides)
        metrics_attention_mask = torch.ones(metrics_embedded.size(0), 1).to(metrics.device)

        # Encoder les métriques avec un transformer pré-entraîné
        metrics_encoder_output = self.metrics_encoder(
            inputs_embeds=metrics_embedded,
            attention_mask=metrics_attention_mask
        )
        metrics_encoded = metrics_encoder_output.last_hidden_state  # [B, 1, 384]

        # Encoder le code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_encoded = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_encoded = self.code_projection(code_encoded)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_encoded,     # Key
            code_encoded      # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# Fonction d'équilibrage avec BorderlineSMOTE
def apply_borderline_smote(X, y, code_lines):
    """
    Applique BorderlineSMOTE pour équilibrer les classes
    """
    print(f"Distribution avant équilibrage: {Counter(y)}")

    # Vérifier qu'il y a assez d'échantillons minoritaires
    min_samples = min(Counter(y).values())
    k_neighbors = min(3, min_samples - 1) if min_samples > 1 else 1

    try:
        smote = BorderlineSMOTE(random_state=42, k_neighbors=k_neighbors, kind='borderline-1')
        X_resampled, y_resampled = smote.fit_resample(X, y)
    except Exception as e:
        print(f"Erreur BorderlineSMOTE: {e}, utilisation de SMOTE standard")
        from imblearn.over_sampling import SMOTE
        smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_resampled, y_resampled = smote.fit_resample(X, y)

    print(f"Distribution après équilibrage: {Counter(y_resampled)}")

    # Adapter la liste de code
    n_original = len(code_lines)
    n_resampled = len(X_resampled)

    if n_resampled > n_original:
        # Mapping pour les indices répliqués
        indices_map = []
        for i in range(n_resampled):
            if i < n_original:
                indices_map.append(i)
            else:
                # Pour les nouveaux échantillons, choisir aléatoirement
                indices_map.append(np.random.randint(0, n_original))

        code_lines_resampled = [code_lines[idx] for idx in indices_map]
    else:
        code_lines_resampled = code_lines[:n_resampled]

    return X_resampled, y_resampled, code_lines_resampled

# Calcul des poids de classe
def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    return weight_tensor

# Prétraitement des métriques
def preprocess_metrics(df):
    """Préprocessing robuste des métriques"""
    if 'name' in df.columns:
        df = df.drop(columns=['name'])

    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    labels = df['bug'].values

    # Traiter les valeurs manquantes
    if features.isnull().any().any():
        print("Valeurs manquantes détectées, remplacement par la médiane")
        features = features.fillna(features.median())

    # Supprimer les features avec variance nulle
    var_threshold = 0.01
    low_var_cols = features.var() < var_threshold
    if low_var_cols.any():
        print(f"Suppression de {low_var_cols.sum()} colonnes avec faible variance")
        features = features.loc[:, ~low_var_cols]

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    print(f"Distribution des classes: {Counter(labels)}")

    return features_scaled, labels, scaler

# Chargement des métriques et du code
def load_code_and_metrics(metrics_path, code_path):
    """Charge les métriques et le code source"""
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []

    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                            code = f.read()
                            code = code.strip()
                            if len(code) > 0:
                                code_lines.append(code)
                    except Exception as e:
                        print(f"Erreur lecture fichier {file}: {e}")
                        continue

    # S'assurer qu'on a au moins autant de fichiers code que de métriques
    if len(code_lines) < len(df_metrics):
        print(f"Attention: {len(code_lines)} fichiers code vs {len(df_metrics)} métriques")
        while len(code_lines) < len(df_metrics):
            if len(code_lines) > 0:
                code_lines.append(code_lines[len(code_lines) % len(code_lines)])
            else:
                code_lines.append("// Empty code file")

    return df_metrics, code_lines

# Préparation du dataset avec BorderlineSMOTE
def prepare_single_dataset(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset avec BorderlineSMOTE"""
    df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
    features, labels, scaler = preprocess_metrics(df_metrics)

    # Assurer la cohérence des tailles
    min_len = min(len(features), len(code_lines), len(labels))
    features = features[:min_len]
    code_lines = code_lines[:min_len]
    labels = labels[:min_len]

    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_lines = apply_borderline_smote(features, labels, code_lines)
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")

    # Tokenisation du code
    try:
        code_tokens = tokenizer(
            code_lines,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        code_tokens = {
            'input_ids': torch.zeros((len(features), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(features), 512), dtype=torch.long)
        }

    return features, code_tokens, labels, scaler

# Préparation de datasets multiples
def prepare_multiple_datasets(metrics_paths, code_paths, tokenizer, apply_balancing=True):
    """Prépare et combine plusieurs datasets"""
    all_features, all_labels, all_code_tokens = [], [], []

    for metrics_path, code_path in zip(metrics_paths, code_paths):
        features, code_tokens, labels, _ = prepare_single_dataset(
            metrics_path, code_path, tokenizer, apply_balancing
        )
        all_features.append(features)
        all_labels.extend(labels)

        # Convertir les tokens pour concaténation
        for i in range(len(features)):
            all_code_tokens.append({
                'input_ids': code_tokens['input_ids'][i],
                'attention_mask': code_tokens['attention_mask'][i]
            })

    # Concaténer toutes les features
    X_combined = np.vstack(all_features)
    y_combined = np.array(all_labels)

    # Reconvertir les tokens
    combined_code_tokens = {
        'input_ids': torch.stack([item['input_ids'] for item in all_code_tokens]),
        'attention_mask': torch.stack([item['attention_mask'] for item in all_code_tokens])
    }

    return X_combined, combined_code_tokens, y_combined

# Évaluation
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# Fonction d'entraînement principale
def train(use_full_pretrained=False):
    """
    Entraînement avec BorderlineSMOTE
    use_full_pretrained: False = Hybride (encoder scratch + decoder préentraîné)
                        True = Full préentraîné (encoder + decoder préentraînés)
    """
    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Définir les chemins
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/ant/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/ant/"

    # Versions pour l'entraînement
    train_versions = ["1.3", "1.4", "1.5", "1.6"]
    metrics_train_paths = [f"{base_metrics_path}ant-{v}.csv" for v in train_versions]
    code_train_paths = [f"{base_code_path}ant-{v}" for v in train_versions]

    # Version pour la validation
    val_version = "1.7"
    metrics_val_path = f"{base_metrics_path}ant-{val_version}.csv"
    code_val_path = f"{base_code_path}ant-{val_version}"

    # Préparer les datasets d'entraînement avec BorderlineSMOTE
    X_train, train_code_tokens, y_train = prepare_multiple_datasets(
        metrics_train_paths, code_train_paths, tokenizer, apply_balancing=True
    )

    # Préparer le dataset de validation (sans équilibrage)
    X_val, code_val_tokens, y_val, _ = prepare_single_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    # Créer les datasets
    train_ds = BugDataset(X_train, train_code_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)

    # Créer les DataLoaders
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Calculer les poids de classe
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    print(f"Poids des classes: {class_weights}")

    # Initialiser le modèle selon le type
    if use_full_pretrained:
        model = FullPretrainedModel().to(Config.DEVICE)
        model_type = "full_pretrained"
        print("Utilisation du modèle avec encoder et decoder pré-entraînés")
    else:
        model = HybridTransBugModel().to(Config.DEVICE)
        model_type = "hybrid"
        print("Utilisation du modèle avec encoder from scratch et decoder pré-entraîné")

    # Optimiseur avec learning rates différenciés
    if use_full_pretrained:
        # Pour le modèle entièrement pré-entraîné, utiliser des LR plus faibles pour les parties pré-entraînées
        optimizer = torch.optim.AdamW([
            {'params': model.metrics_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters()
                       if 'metrics_encoder' not in n and 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)
    else:
        # Pour le modèle hybride, seul le decoder est pré-entraîné
        optimizer = torch.optim.AdamW([
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters() if 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=True
    )

    # Loss avec pondération et label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0

    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        print(f"Epoch {epoch+1}:")
        print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
        print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
        print(f"  Val: Sensitivité={sens:.4f}, Spécificité={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            joblib.dump(cm, f"{Config.SAVE_PATH}conf_matrix_{model_type}.pkl")
            joblib.dump(val_preds, f"{Config.SAVE_PATH}val_preds_{model_type}.pkl")
            joblib.dump(val_labels, f"{Config.SAVE_PATH}val_labels_{model_type}.pkl")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 7:
            print("Early stopping déclenché")
            break

    print(f"\n{model_type.upper()} - Meilleur F1 score: {best_f1:.4f}")
    print("Classification Report:")
    if len(set(val_labels)) > 1:
        print(classification_report(val_labels, val_preds))
    else:
        print("Une seule classe présente dans les prédictions")

    # Visualisations
    plt.figure(figsize=(15, 5))

    # Courbe 1: Loss
    plt.subplot(1, 3, 1)
    epochs_range = range(1, len(train_losses) + 1)
    plt.plot(epochs_range, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs_range, val_losses, 'r-', label='Val Loss')
    plt.title(f'{model_type.capitalize()} - Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Courbe 2: Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
    plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
    plt.title(f'{model_type.capitalize()} - Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Courbe 3: F1-score et G-mean
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, f1_scores, 'g-', label='F1 Score')
    plt.plot(epochs_range, gmean_scores, 'purple', label='G-Mean')
    plt.title(f'{model_type.capitalize()} - F1-Score and G-Mean Evolution')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}learning_curves_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

        # Confusion matrix visualization
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.savefig(f"{Config.SAVE_PATH}conf_matrix_{model_type}.png")
    plt.show()

    return {
        'model_type': model_type,
        'best_f1': best_f1,
        'val_accuracy': val_accuracies[-1],
        'train_accuracy': train_accuracies[-1],
        'val_loss': val_losses[-1],
        'train_loss': train_losses[-1],
        'gmean': gmean_scores[-1],
    }


if __name__ == "__main__":
    # Entraîner le modèle hybride (encodeur from scratch + décodeur pré-entraîné)
    hybrid_results = train(use_full_pretrained=False)

    # Entraîner le modèle entièrement pré-entraîné
    full_results = train(use_full_pretrained=True)

    # Comparer les résultats
    print("\n===== COMPARAISON DES PERFORMANCES =====")
    print(f"Modèle hybride: F1={hybrid_results['best_f1']:.4f}, G-Mean={hybrid_results['gmean']:.4f}")
    print(f"Modèle full-pretrained: F1={full_results['best_f1']:.4f}, G-Mean={full_results['gmean']:.4f}")

    # Afficher un résumé des résultats en tableau
    results_df = pd.DataFrame([hybrid_results, full_results])
    print("\n", results_df[['model_type', 'best_f1', 'gmean', 'val_accuracy', 'train_accuracy']])

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, f1_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE, SMOTE
from transformers import RobertaTokenizer, RobertaModel, RobertaConfig
from scipy.stats import gmean
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from collections import Counter

# Configuration
class Config:
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    EPOCHS = 25
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-5
    EMBEDDING_DIM = 384
    N_HEADS = 8
    NUM_LAYERS = 2
    DROPOUT = 0.3
    NUM_CLASSES = 2
    SAVE_PATH = '/content/drive/MyDrive/bug/results/'

    # Paramètres pour l'équilibrage
    LABEL_SMOOTHING = 0.1
    GRADIENT_CLIP = 1.0

    os.makedirs(SAVE_PATH, exist_ok=True)

# Dataset personnalisé
class BugDataset(Dataset):
    def __init__(self, metrics, code_tokens, labels):
        self.metrics = torch.FloatTensor(metrics)
        self.code_tokens = code_tokens
        self.labels = torch.LongTensor(labels)

        # Vérifier la cohérence des tailles
        assert len(self.metrics) == len(self.labels), "Mismatch entre metrics et labels"
        assert len(self.code_tokens['input_ids']) == len(self.labels), "Mismatch entre code_tokens et labels"

    def __len__(self):
        return len(self.metrics)

    def __getitem__(self, idx):
        return self.metrics[idx], {
            'input_ids': self.code_tokens['input_ids'][idx],
            'attention_mask': self.code_tokens['attention_mask'][idx]
        }, self.labels[idx]

# STRATÉGIE 1: Modèle hybride (encoder from scratch + decoder pré-entraîné)
class HybridTransBugModel(nn.Module):
    def __init__(self, input_metric_dim=None):
        super().__init__()
        # Projection des métriques vers la dimension d'embedding
        self.metric_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM) if input_metric_dim is None else nn.Linear(input_metric_dim, Config.EMBEDDING_DIM),
            nn.BatchNorm1d(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Encoder transformer FROM SCRATCH pour les métriques
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=Config.EMBEDDING_DIM,
            nhead=Config.N_HEADS,
            dropout=Config.DROPOUT,
            activation='gelu',
            batch_first=True,
            norm_first=True
        )
        self.metrics_encoder = nn.TransformerEncoder(encoder_layer, num_layers=Config.NUM_LAYERS)

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        roberta_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(roberta_config)

        # Projection pour aligner les dimensions RoBERTa -> Config.EMBEDDING_DIM
        self.code_projection = nn.Linear(roberta_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention pour combiner les représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Traitement des métriques avec encoder from scratch
        metrics_proj = self.metric_projection(metrics)
        metrics_proj = metrics_proj.unsqueeze(1)  # [B, 1, 384]
        metrics_encoded = self.metrics_encoder(metrics_proj)  # [B, 1, 384]

        # Traitement du code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_embed = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_embed = self.code_projection(code_embed)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_embed,      # Key
            code_embed       # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# STRATÉGIE 2: Modèle entièrement pré-entraîné (encoder et decoder pré-entraînés)
class FullPretrainedModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder PRÉ-ENTRAÎNÉ pour les métriques (adaptation de RoBERTa)
        self.metrics_encoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.metrics_encoder_config.hidden_size = Config.EMBEDDING_DIM
        self.metrics_encoder = RobertaModel(self.metrics_encoder_config)

        # Projection pour les métriques numériques vers le format attendu par transformer
        self.metrics_projection = nn.Sequential(
            nn.LazyLinear(Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.Dropout(Config.DROPOUT)
        )

        # Decoder PRÉ-ENTRAÎNÉ RoBERTa pour le code
        self.code_decoder_config = RobertaConfig.from_pretrained("microsoft/codebert-base")
        self.code_decoder = RobertaModel(self.code_decoder_config)

        # Projection pour aligner les dimensions
        self.code_projection = nn.Linear(self.code_decoder_config.hidden_size, Config.EMBEDDING_DIM)

        # Couche d'attention croisée pour combiner les deux représentations
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=Config.EMBEDDING_DIM,
            num_heads=Config.N_HEADS,
            dropout=Config.DROPOUT,
            batch_first=True
        )

        # Fusion des représentations
        self.fusion = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM * 2, Config.EMBEDDING_DIM),
            nn.LayerNorm(Config.EMBEDDING_DIM),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT)
        )

        # Classifier final
        self.classifier = nn.Sequential(
            nn.Linear(Config.EMBEDDING_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(Config.DROPOUT),
            nn.Linear(128, Config.NUM_CLASSES)
        )

    def forward(self, metrics, code_tokens):
        # Projeter les métriques et les adapter au format attendu
        metrics_embedded = self.metrics_projection(metrics).unsqueeze(1)  # [B, 1, 384]

        # Créer un masque factice pour les métriques (toutes valides)
        metrics_attention_mask = torch.ones(metrics_embedded.size(0), 1).to(metrics.device)

        # Encoder les métriques avec un transformer pré-entraîné
        metrics_encoder_output = self.metrics_encoder(
            inputs_embeds=metrics_embedded,
            attention_mask=metrics_attention_mask
        )
        metrics_encoded = metrics_encoder_output.last_hidden_state  # [B, 1, 384]

        # Encoder le code source avec decoder pré-entraîné
        code_outputs = self.code_decoder(**code_tokens)
        code_encoded = code_outputs.last_hidden_state  # [B, seq_len, 768]
        code_encoded = self.code_projection(code_encoded)  # [B, seq_len, 384]

        # Attention croisée entre métriques et code
        attn_output, _ = self.cross_attention(
            metrics_encoded,  # Query
            code_encoded,     # Key
            code_encoded      # Value
        )

        # Fusion avec skip connection
        combined = torch.cat([metrics_encoded[:, 0, :], attn_output[:, 0, :]], dim=-1)
        fused = self.fusion(combined)

        # Classification
        return self.classifier(fused)

# Fonction d'équilibrage avec BorderlineSMOTE
def apply_borderline_smote(X, y, code_lines):
    """
    Applique BorderlineSMOTE pour équilibrer les classes
    """
    print(f"Distribution avant équilibrage: {Counter(y)}")

    # Vérifier qu'il y a assez d'échantillons minoritaires
    min_samples = min(Counter(y).values())
    k_neighbors = min(3, min_samples - 1) if min_samples > 1 else 1

    try:
        smote = BorderlineSMOTE(random_state=42, k_neighbors=k_neighbors, kind='borderline-1')
        X_resampled, y_resampled = smote.fit_resample(X, y)
        print("✓ BorderlineSMOTE appliqué avec succès")
    except Exception as e:
        print(f"Erreur BorderlineSMOTE: {e}, utilisation de SMOTE standard")
        smote = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_resampled, y_resampled = smote.fit_resample(X, y)

    print(f"Distribution après équilibrage: {Counter(y_resampled)}")

    # Adapter la liste de code
    n_original = len(code_lines)
    n_resampled = len(X_resampled)

    if n_resampled > n_original:
        # Mapping pour les indices répliqués
        indices_map = []
        for i in range(n_resampled):
            if i < n_original:
                indices_map.append(i)
            else:
                # Pour les nouveaux échantillons, choisir aléatoirement
                indices_map.append(np.random.randint(0, n_original))

        code_lines_resampled = [code_lines[idx] for idx in indices_map]
    else:
        code_lines_resampled = code_lines[:n_resampled]

    return X_resampled, y_resampled, code_lines_resampled

# Calcul des poids de classe
def calculate_class_weights(labels):
    """Calcule les poids de classe pour équilibrer la loss"""
    class_counts = Counter(labels)
    total_samples = len(labels)
    n_classes = len(class_counts)

    weights = {}
    for class_id, count in class_counts.items():
        weights[class_id] = total_samples / (n_classes * count)

    # Convertir en tensor PyTorch
    weight_tensor = torch.FloatTensor([weights[i] for i in sorted(weights.keys())])
    return weight_tensor

# Prétraitement des métriques
def preprocess_metrics(df):
    """Préprocessing robuste des métriques"""
    if 'name' in df.columns:
        df = df.drop(columns=['name'])

    df['bug'] = (df['bug'] > 0).astype(int)
    features = df.drop(columns=['bug'])
    labels = df['bug'].values

    # Traiter les valeurs manquantes
    if features.isnull().any().any():
        print("Valeurs manquantes détectées, remplacement par la médiane")
        features = features.fillna(features.median())

    # Supprimer les features avec variance nulle
    var_threshold = 0.01
    low_var_cols = features.var() < var_threshold
    if low_var_cols.any():
        print(f"Suppression de {low_var_cols.sum()} colonnes avec faible variance")
        features = features.loc[:, ~low_var_cols]

    # Normalisation
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features)

    print(f"Distribution des classes: {Counter(labels)}")

    return features_scaled, labels, scaler

# Chargement des métriques et du code
def load_code_and_metrics(metrics_path, code_path):
    """Charge les métriques et le code source"""
    df_metrics = pd.read_csv(metrics_path)
    code_lines = []

    if os.path.exists(code_path):
        for root, dirs, files in os.walk(code_path):
            for file in files:
                if file.endswith('.java'):
                    try:
                        with open(os.path.join(root, file), 'r', encoding='utf-8', errors='ignore') as f:
                            code = f.read()
                            code = code.strip()
                            if len(code) > 0:
                                code_lines.append(code)
                    except Exception as e:
                        print(f"Erreur lecture fichier {file}: {e}")
                        continue

    # S'assurer qu'on a au moins autant de fichiers code que de métriques
    if len(code_lines) < len(df_metrics):
        print(f"Attention: {len(code_lines)} fichiers code vs {len(df_metrics)} métriques")
        while len(code_lines) < len(df_metrics):
            if len(code_lines) > 0:
                code_lines.append(code_lines[len(code_lines) % len(code_lines)])
            else:
                code_lines.append("// Empty code file")

    return df_metrics, code_lines

# Préparation du dataset avec BorderlineSMOTE
def prepare_single_dataset(metrics_path, code_path, tokenizer, apply_balancing=True):
    """Prépare un dataset avec BorderlineSMOTE"""
    df_metrics, code_lines = load_code_and_metrics(metrics_path, code_path)
    features, labels, scaler = preprocess_metrics(df_metrics)

    # Assurer la cohérence des tailles
    min_len = min(len(features), len(code_lines), len(labels))
    features = features[:min_len]
    code_lines = code_lines[:min_len]
    labels = labels[:min_len]

    if apply_balancing and len(set(labels)) > 1:
        try:
            features, labels, code_lines = apply_borderline_smote(features, labels, code_lines)
        except Exception as e:
            print(f"Erreur lors de l'équilibrage: {e}")
            print("Utilisation des données originales")

    # Tokenisation du code
    try:
        code_tokens = tokenizer(
            code_lines,
            padding=True,
            truncation=True,
            return_tensors='pt',
            max_length=512
        )
    except Exception as e:
        print(f"Erreur lors de la tokenisation: {e}")
        code_tokens = {
            'input_ids': torch.zeros((len(features), 512), dtype=torch.long),
            'attention_mask': torch.zeros((len(features), 512), dtype=torch.long)
        }

    return features, code_tokens, labels, scaler

# Préparation de datasets multiples
def prepare_multiple_datasets(metrics_paths, code_paths, tokenizer, apply_balancing=True):
    """Prépare et combine plusieurs datasets"""
    all_features, all_labels, all_code_tokens = [], [], []

    for metrics_path, code_path in zip(metrics_paths, code_paths):
        features, code_tokens, labels, _ = prepare_single_dataset(
            metrics_path, code_path, tokenizer, apply_balancing
        )
        all_features.append(features)
        all_labels.extend(labels)

        # Convertir les tokens pour concaténation
        for i in range(len(features)):
            all_code_tokens.append({
                'input_ids': code_tokens['input_ids'][i],
                'attention_mask': code_tokens['attention_mask'][i]
            })

    # Concaténer toutes les features
    X_combined = np.vstack(all_features)
    y_combined = np.array(all_labels)

    # Reconvertir les tokens
    combined_code_tokens = {
        'input_ids': torch.stack([item['input_ids'] for item in all_code_tokens]),
        'attention_mask': torch.stack([item['attention_mask'] for item in all_code_tokens])
    }

    return X_combined, combined_code_tokens, y_combined

# Évaluation
def evaluate(model, dataloader, criterion=None):
    """Évaluation robuste du modèle"""
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0

    with torch.no_grad():
        for metrics, code_tokens, labels in dataloader:
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)

            if criterion is not None:
                loss = criterion(outputs, labels)
                total_loss += loss.item() * labels.size(0)

            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculer les métriques
    cm = confusion_matrix(all_labels, all_preds, labels=[0, 1])

    try:
        f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    except:
        f1 = 0.0

    accuracy = accuracy_score(all_labels, all_preds)

    # Calcul robuste de sensibilité et spécificité
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        sens = tp / (tp + fn + 1e-8)
        spec = tn / (tn + fp + 1e-8)

        if sens > 0 and spec > 0:
            g_mean = gmean([sens, spec])
        else:
            g_mean = 0.0
    else:
        sens = spec = g_mean = 0.0

    avg_loss = total_loss / len(all_labels) if criterion is not None and len(all_labels) > 0 else None

    return cm, f1, sens, spec, g_mean, accuracy, avg_loss, all_preds, all_labels

# Fonction d'entraînement principale
def train_model(model_type="hybrid"):
    """
    Entraînement avec BorderlineSMOTE
    model_type: "hybrid" = Encoder from scratch + Decoder pré-entraîné
                "full_pretrained" = Encoder et Decoder pré-entraînés
    """
    print(f"\n{'='*50}")
    print(f"ENTRAÎNEMENT: {model_type.upper()}")
    print(f"{'='*50}")

    tokenizer = RobertaTokenizer.from_pretrained("microsoft/codebert-base")

    # Définir les chemins
    base_metrics_path = "/content/drive/MyDrive/bug/bug-data/ant/"
    base_code_path = "/content/drive/MyDrive/bug/bug-data/source-code/source-code/ant/"

    # Versions pour l'entraînement
    train_versions = ["1.3", "1.4", "1.5", "1.6"]
    metrics_train_paths = [f"{base_metrics_path}ant-{v}.csv" for v in train_versions]
    code_train_paths = [f"{base_code_path}ant-{v}" for v in train_versions]

    # Version pour la validation
    val_version = "1.7"
    metrics_val_path = f"{base_metrics_path}ant-{val_version}.csv"
    code_val_path = f"{base_code_path}ant-{val_version}"

    # Préparer les datasets d'entraînement avec BorderlineSMOTE
    print("Préparation des données d'entraînement avec BorderlineSMOTE...")
    X_train, train_code_tokens, y_train = prepare_multiple_datasets(
        metrics_train_paths, code_train_paths, tokenizer, apply_balancing=True
    )

    # Préparer le dataset de validation (sans équilibrage)
    print("Préparation des données de validation...")
    X_val, code_val_tokens, y_val, _ = prepare_single_dataset(
        metrics_val_path, code_val_path, tokenizer, apply_balancing=False
    )

    # Créer les datasets
    train_ds = BugDataset(X_train, train_code_tokens, y_train)
    val_ds = BugDataset(X_val, code_val_tokens, y_val)

    # Créer les DataLoaders
    train_loader = DataLoader(train_ds, batch_size=Config.BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=Config.BATCH_SIZE)

    # Calculer les poids de classe
    class_weights = calculate_class_weights(y_train).to(Config.DEVICE)
    print(f"Poids des classes: {class_weights}")

    # Initialiser le modèle selon le type
    if model_type == "full_pretrained":
        model = FullPretrainedModel().to(Config.DEVICE)
        print("✓ Modèle avec ENCODER et DECODER pré-entraînés")
    else:
        model = HybridTransBugModel().to(Config.DEVICE)
        print("✓ Modèle avec ENCODER from scratch et DECODER pré-entraîné")

    # Optimiseur avec learning rates différenciés
    if model_type == "full_pretrained":
        # Pour le modèle entièrement pré-entraîné, utiliser des LR plus faibles pour les parties pré-entraînées
        optimizer = torch.optim.AdamW([
            {'params': model.metrics_encoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters()
                       if 'metrics_encoder' not in n and 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)
    else:
        # Pour le modèle hybride, seul le decoder est pré-entraîné
        optimizer = torch.optim.AdamW([
            {'params': model.code_decoder.parameters(), 'lr': Config.LEARNING_RATE * 0.1},
            {'params': [p for n, p in model.named_parameters() if 'code_decoder' not in n],
             'lr': Config.LEARNING_RATE}
        ], weight_decay=0.01)

    # Scheduler
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=True
    )

    # Loss avec pondération et label smoothing
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=Config.LABEL_SMOOTHING)

    # Listes pour le monitoring
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []
    f1_scores, gmean_scores = [], []

    best_f1 = 0
    patience_counter = 0

    for epoch in range(Config.EPOCHS):
        # Entraînement
        model.train()
        train_loss = 0
        train_preds, train_labels_list = [], []

        for batch_idx, (metrics, code_tokens, labels) in enumerate(train_loader):
            metrics = metrics.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE)

            code_tokens_dict = {
                'input_ids': code_tokens['input_ids'].to(Config.DEVICE),
                'attention_mask': code_tokens['attention_mask'].to(Config.DEVICE)
            }

            outputs = model(metrics, code_tokens_dict)
            loss = criterion(outputs, labels)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), Config.GRADIENT_CLIP)

            optimizer.step()

            train_loss += loss.item() * labels.size(0)
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())

        # Métriques d'entraînement
        train_loss /= len(train_preds)
        train_acc = accuracy_score(train_labels_list, train_preds)
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        # Évaluation
        cm, f1, sens, spec, gmean, val_acc, val_loss, val_preds, val_labels = evaluate(model, val_loader, criterion)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        f1_scores.append(f1)
        gmean_scores.append(gmean)

        # Scheduler step
        scheduler.step(f1)

        print(f"Epoch {epoch+1}/{Config.EPOCHS}:")
        print(f"  Train: Loss={train_loss:.4f}, Accuracy={train_acc:.4f}")
        print(f"  Val: Loss={val_loss:.4f}, Accuracy={val_acc:.4f}, F1={f1:.4f}, G-Mean={gmean:.4f}")
        print(f"  Val: Sensitivité={sens:.4f}, Spécificité={spec:.4f}")

        # Sauvegarde du meilleur modèle
        if f1 > best_f1:
            best_f1 = f1
            patience_counter = 0
            torch.save(model.state_dict(), f"{Config.SAVE_PATH}best_model_{model_type}.pt")
            joblib.dump(cm, f"{Config.SAVE_PATH}conf_matrix_{model_type}.pkl")
            joblib.dump(val_preds, f"{Config.SAVE_PATH}val_preds_{model_type}.pkl")
            joblib.dump(val_labels, f"{Config.SAVE_PATH}val_labels_{model_type}.pkl")
            print(f"  *** Nouveau meilleur F1: {f1:.4f} ***")
        else:
            patience_counter += 1

        # Early stopping
        if patience_counter >= 7:
            print("Early stopping déclenché")
            break

    print(f"\n{model_type.upper()} - Meilleur F1 score: {best_f1:.4f}")

    # Visualisations
    plt.figure(figsize=(15, 5))

    # Courbe 1: Loss
    plt.subplot(1, 3, 1)
    epochs_range = range(1, len(train_losses) + 1)
    plt.plot(epochs_range, train_losses, 'b-', label='Train Loss')
    plt.plot(epochs_range, val_losses, 'r-', label='Val Loss')
    plt.title(f'{model_type.capitalize()} - Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # Courbe 2: Accuracy
    plt.subplot(1, 3, 2)
    plt.plot(epochs_range, train_accuracies, 'b-', label='Train Accuracy')
    plt.plot(epochs_range, val_accuracies, 'r-', label='Val Accuracy')
    plt.title(f'{model_type.capitalize()} - Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Courbe 3: F1-score et G-mean
    plt.subplot(1, 3, 3)
    plt.plot(epochs_range, f1_scores, 'g-', label='F1-Score')
    plt.plot(epochs_range, gmean_scores, 'm-', label='G-Mean')
    plt.title(f'{model_type.capitalize()} - F1-Score and G-Mean')
    plt.xlabel('Epochs')
    plt.ylabel('Score')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}training_curves_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Matrice de confusion finale
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Bug', 'Bug'],
                yticklabels=['Non-Bug', 'Bug'])
    plt.title(f'{model_type.capitalize()} - Confusion Matrix (Best Model)')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.savefig(f"{Config.SAVE_PATH}confusion_matrix_{model_type}.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Sauvegarder les métriques
    results = {
        'model_type': model_type,
        'best_f1': best_f1,
        'final_accuracy': val_accuracies[-1] if val_accuracies else 0,
        'final_sensitivity': sens,
        'final_specificity': spec,
        'final_gmean': gmean,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'f1_scores': f1_scores,
        'gmean_scores': gmean_scores
    }

    joblib.dump(results, f"{Config.SAVE_PATH}training_results_{model_type}.pkl")

    return model, results

def compare_models():
    """Compare les deux stratégies de modèles"""
    print("\n" + "="*60)
    print("COMPARAISON DES DEUX STRATÉGIES")
    print("="*60)

    # Entraîner le modèle hybride
    print("\n1. STRATÉGIE HYBRIDE (Encoder from scratch + Decoder pré-entraîné)")
    model_hybrid, results_hybrid = train_model("hybrid")

    # Entraîner le modèle entièrement pré-entraîné
    print("\n2. STRATÉGIE ENTIÈREMENT PRÉ-ENTRAÎNÉE (Encoder et Decoder pré-entraînés)")
    model_full, results_full = train_model("full_pretrained")

    # Comparaison des résultats
    print("\n" + "="*60)
    print("RÉSULTATS COMPARATIFS")
    print("="*60)

    print(f"{'Métrique':<20} {'Hybride':<15} {'Full Pretrained':<15} {'Meilleur':<15}")
    print("-" * 70)

    metrics_comparison = [
        ('F1-Score', results_hybrid['best_f1'], results_full['best_f1']),
        ('Accuracy', results_hybrid['final_accuracy'], results_full['final_accuracy']),
        ('Sensitivity', results_hybrid['final_sensitivity'], results_full['final_sensitivity']),
        ('Specificity', results_hybrid['final_specificity'], results_full['final_specificity']),
        ('G-Mean', results_hybrid['final_gmean'], results_full['final_gmean'])
    ]

    for metric_name, hybrid_score, full_score in metrics_comparison:
        better = "Hybride" if hybrid_score > full_score else "Full Pretrained" if full_score > hybrid_score else "Égalité"
        print(f"{metric_name:<20} {hybrid_score:<15.4f} {full_score:<15.4f} {better:<15}")

    # Graphique comparatif
    plt.figure(figsize=(12, 8))

    # Comparaison F1-Score au fil des époques
    plt.subplot(2, 2, 1)
    epochs_hybrid = range(1, len(results_hybrid['f1_scores']) + 1)
    epochs_full = range(1, len(results_full['f1_scores']) + 1)
    plt.plot(epochs_hybrid, results_hybrid['f1_scores'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['f1_scores'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison F1-Score')
    plt.xlabel('Epochs')
    plt.ylabel('F1-Score')
    plt.legend()
    plt.grid(True)

    # Comparaison G-Mean
    plt.subplot(2, 2, 2)
    plt.plot(epochs_hybrid, results_hybrid['gmean_scores'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['gmean_scores'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison G-Mean')
    plt.xlabel('Epochs')
    plt.ylabel('G-Mean')
    plt.legend()
    plt.grid(True)

    # Comparaison Accuracy
    plt.subplot(2, 2, 3)
    plt.plot(epochs_hybrid, results_hybrid['val_accuracies'], 'b-', label='Hybride', linewidth=2)
    plt.plot(epochs_full, results_full['val_accuracies'], 'r-', label='Full Pretrained', linewidth=2)
    plt.title('Comparaison Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    # Graphique en barres des métriques finales
    plt.subplot(2, 2, 4)
    metrics_names = ['F1-Score', 'Accuracy', 'Sensitivity', 'Specificity', 'G-Mean']
    hybrid_values = [results_hybrid['best_f1'], results_hybrid['final_accuracy'],
                    results_hybrid['final_sensitivity'], results_hybrid['final_specificity'],
                    results_hybrid['final_gmean']]
    full_values = [results_full['best_f1'], results_full['final_accuracy'],
                  results_full['final_sensitivity'], results_full['final_specificity'],
                  results_full['final_gmean']]

    x = np.arange(len(metrics_names))
    width = 0.35

    plt.bar(x - width/2, hybrid_values, width, label='Hybride', alpha=0.8)
    plt.bar(x + width/2, full_values, width, label='Full Pretrained', alpha=0.8)

    plt.title('Comparaison des Métriques Finales')
    plt.xlabel('Métriques')
    plt.ylabel('Score')
    plt.xticks(x, metrics_names, rotation=45)
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f"{Config.SAVE_PATH}models_comparison.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Recommandation
    print("\n" + "="*60)
    print("RECOMMANDATION")
    print("="*60)

    if results_hybrid['best_f1'] > results_full['best_f1']:
        best_model = "Hybride"
        best_results = results_hybrid
    else:
        best_model = "Full Pretrained"
        best_results = results_full

    print(f"✓ Le modèle {best_model} obtient les meilleurs résultats")
    print(f"  - F1-Score: {best_results['best_f1']:.4f}")
    print(f"  - G-Mean: {best_results['final_gmean']:.4f}")
    print(f"  - Accuracy: {best_results['final_accuracy']:.4f}")

    # Analyse des avantages
    print(f"\nAnalyse des stratégies:")
    print("• Modèle Hybride:")
    print("  - Encoder from scratch pour les métriques")
    print("  - Decoder pré-entraîné pour le code source")
    print("  - Plus flexible pour l'adaptation aux métriques spécifiques")

    print("• Modèle Full Pretrained:")
    print("  - Encoder et Decoder pré-entraînés")
    print("  - Tire parti de plus de connaissances pré-entraînées")
    print("  - Peut nécessiter moins d'entraînement")

    return best_model, best_results

def main():
    """Fonction principale pour lancer la comparaison"""
    print("🚀 Démarrage de l'entraînement et comparaison des modèles de détection de bugs")
    print("Utilisation de BorderlineSMOTE pour l'équilibrage des classes")

    try:
        best_model, best_results = compare_models()

        print(f"\n🎉 Entraînement terminé avec succès!")
        print(f"Meilleur modèle: {best_model}")
        print(f"Résultats sauvegardés dans: {Config.SAVE_PATH}")

    except Exception as e:
        print(f"❌ Erreur lors de l'entraînement: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()